In [ ]:
from google.colab import drive
from pathlib import Path
import os

drive.mount("/content/drive")

PROJECT_DIR = Path("/content/drive/MyDrive/Colab Notebooks/Adv_Object_Tracking_and_Detection_in_Video_Streams")
assert (PROJECT_DIR / "src").is_dir(), "Check PROJECT_DIR path"
os.chdir(PROJECT_DIR)

print("Working directory:", Path.cwd())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Working directory: /content/drive/MyDrive/Colab Notebooks/CV_asgn2


In [2]:
!pip -q install kagglehub PyYAML scipy opencv-python-headless motmetrics "torchmetrics[detection]" pycocotools

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.5/161.5 kB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 59.3 MB/s eta 0:00:00


In [3]:
import kagglehub
path = kagglehub.dataset_download("wenhoujinjust/mot-17")

Using Colab cache for faster access to the 'mot-17' dataset.


In [4]:
print("Kaggle download location:", path)

Kaggle download location: /kaggle/input/mot-17


In [5]:
download_dir = Path(path)
candidates = [download_dir] + [p for p in download_dir.rglob("MOT17") if p.is_dir()]

dataset_root = next(
    p for p in candidates
    if (p / "train").is_dir() and (p / "test").is_dir()
)

print("Using dataset:", dataset_root)

Using dataset: /kaggle/input/mot-17/MOT17


In [6]:
import torch
import yaml

with open("config.yaml", encoding="utf-8") as file:
    config = yaml.safe_load(file)

config["paths"]["dataset_root"] = str(dataset_root)
config["paths"]["output_root"] = str(PROJECT_DIR / "results")
config["paths"]["checkpoint_path"] = str(
    PROJECT_DIR / "results" / "checkpoints" / "fasterrcnn_best.pt"
)

config["runtime"]["device"] = "cuda" if torch.cuda.is_available() else "cpu"
config["training"]["batch_size"] = 2 if torch.cuda.is_available() else 1
config["training"]["epochs"] = 4
config["training"]["num_workers"] = 0

with open("config_colab.yaml", "w", encoding="utf-8") as file:
    yaml.safe_dump(config, file, sort_keys=False)

print("GPU available:", torch.cuda.is_available())

GPU available: True


In [7]:
!python -m scripts.check_setup --config config_colab.yaml

Setup check passed
Dataset root: /kaggle/input/mot-17/MOT17
Validation sequence: MOT17-05-FRCNN
Frames: 837
Output root: /content/drive/MyDrive/Colab Notebooks/CV_asgn2/results


In [8]:
!python -m src.train_detector --config config_colab.yaml

Device: cuda; training frames: 435; validation frames: 168
Downloading: "https://download.pytorch.org/models/fasterrcnn_resnet50_fpn_coco-258fb6c6.pth" to /root/.cache/torch/hub/checkpoints/fasterrcnn_resnet50_fpn_coco-258fb6c6.pth
100% 160M/160M [00:01<00:00, 158MB/s]
{'epoch': 1, 'train_loss': 0.8368912853232218, 'seconds': 168.926387551, 'map': 0.3840031921863556, 'map_50': 0.7391573190689087, 'map_75': 0.3562277555465698, 'map_small': 0.07592164725065231, 'map_medium': 0.3313532769680023, 'map_large': 0.44973084330558777, 'mar_1': 0.08698481321334839, 'mar_10': 0.42386117577552795, 'mar_100': 0.46008676290512085, 'mar_small': 0.22575756907463074, 'mar_medium': 0.40337079763412476, 'mar_large': 0.5190293788909912, 'map_per_class': -1.0, 'mar_100_per_class': -1.0, 'classes': 1.0}
Saved best checkpoint: /content/drive/MyDrive/Colab Notebooks/CV_asgn2/results/checkpoints/fasterrcnn_best.pt
{'epoch': 2, 'train_loss': 0.6712609756430354, 'seconds': 168.41592883500005, 'map': 0.3797081410

In [9]:
!python -m src.evaluate_detector --config config_colab.yaml

Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth
100% 97.8M/97.8M [00:00<00:00, 195MB/s]
{'map': 0.3921828866004944, 'map_50': 0.7305525541305542, 'map_75': 0.37316665053367615, 'map_small': 0.06360973417758942, 'map_medium': 0.3382932245731354, 'map_large': 0.45843225717544556, 'mar_1': 0.08720173686742783, 'mar_10': 0.43022415041923523, 'mar_100': 0.455025315284729, 'mar_small': 0.13636364042758942, 'mar_medium': 0.40430712699890137, 'mar_large': 0.5185185074806213, 'map_per_class': -1.0, 'mar_100_per_class': -1.0, 'classes': 1.0, 'sequence': 'MOT17-05-FRCNN', 'checkpoint': '/content/drive/MyDrive/Colab Notebooks/CV_asgn2/results/checkpoints/fasterrcnn_best.pt', 'frames': 168}
Saved /content/drive/MyDrive/Colab Notebooks/CV_asgn2/results/detection_metrics.json


In [10]:
!python -m src.run_tracking --config config_colab.yaml --max-frames 300

Processed 50/300 frames
Processed 100/300 frames
Processed 150/300 frames
Processed 200/300 frames
Processed 250/300 frames
Processed 300/300 frames
Saved /content/drive/MyDrive/Colab Notebooks/CV_asgn2/results/tracking_MOT17-05-FRCNN/tracking_results.txt


In [11]:
!python -m src.evaluate_tracking --config config_colab.yaml --predictions "/content/drive/MyDrive/Colab Notebooks/CV_asgn2/results/tracking_MOT17-05-FRCNN/tracking_results.txt"

{'num_frames': 837, 'mota': 0.17478675726471016, 'motp': 0.2419744799479413, 'idf1': 0.28023598820059, 'precision': 0.8318397469688983, 'recall': 0.22813358392366634, 'num_switches': 50, 'mostly_tracked': 3, 'mostly_lost': 100, 'sequence': 'MOT17-05-FRCNN', 'iou_threshold': 0.5, 'predictions': '/content/drive/MyDrive/Colab Notebooks/CV_asgn2/results/tracking_MOT17-05-FRCNN/tracking_results.txt'}
Saved /content/drive/MyDrive/Colab Notebooks/CV_asgn2/results/tracking_MOT17-05-FRCNN/tracking_metrics.json
